# Pre-env probe - Week 20 dependency install check

Throwaway notebook. It installs the SUPERSET of every Week 20 notebook's
dependencies and then imports + version-checks them. If this runs clean on
the course cluster, the pins are safe to put in the real Week 20 notebooks.

DBR 15.4 LTS ships numpy 1.23.5 / pandas 1.5.3 and a pyarrow compiled
against numpy 1.x. numpy and pandas are pinned <2 so no transitive
dependency can bump them and crash the kernel.

In [ ]:
# Superset install: numpy/pandas pinned <2 FIRST so pip resolves them before
# any package that would otherwise pull numpy 2.x. The course cluster is
# PLAIN DBR 15.4 LTS (not the ML runtime - the Shared Compute policy forbids
# ML runtimes), so mlflow does NOT ship with it. We install mlflow-skinny:
# the lightweight tracking client, which does NOT drag numpy/pandas/scipy and
# so cannot re-trigger the numpy 2.x kernel crash. pyspark IS cluster-provided.
%pip install --quiet \
  "numpy<2" "pandas<2" \
  "boto3>=1.36" \
  "sagemaker>=2.230,<3" \
  "strands-agents>=1.37,<2" "strands-agents-tools>=0.2" \
  "litellm>=1.50" \
  "langfuse>=2.50,<3" \
  "mlflow-skinny>=2.13,<3" \
  "requests>=2.31"
dbutils.library.restartPython()

In [ ]:
# Verify versions after the kernel restart. Use importlib.metadata, never
# pkg.__version__. numpy MUST be < 2 or the kernel pyarrow is broken.
from importlib.metadata import version

pkgs = [
    "numpy", "pandas", "boto3", "sagemaker",
    "strands-agents", "strands-agents-tools",
    "litellm", "langfuse", "mlflow-skinny", "requests",
]
for p in pkgs:
    try:
        print(f"{p:24s} {version(p)}")
    except Exception as e:
        print(f"{p:24s} NOT INSTALLED ({e})")

import numpy as _np
assert _np.__version__.startswith("1."), f"numpy is {_np.__version__} - must be 1.x"
print("\nnumpy is 1.x - OK")

In [ ]:
# Section 1 - imports load + per-student auth.
# Confirms every installed library imports, and builds the AWS clients the
# rest of the preflight uses. boto3 must be new enough for bedrock-runtime
# (the original UnknownServiceError).
import os
import io
import json
import time
import boto3
import sagemaker
import litellm
import langfuse
import pandas
import pyarrow
from strands import Agent

_results = []
def check(name, fn):
    """Run one preflight check; record PASS/FAIL, never abort the notebook."""
    try:
        detail = fn()
        line = f"PASS  {name}" + (f"  ({detail})" if detail else "")
        _results.append((True, name))
    except Exception as e:
        line = f"FAIL  {name}  -> {type(e).__name__}: {e}"
        _results.append((False, name))
    print(line)

# Per-student secret scope, same derivation the course notebooks use.
_user = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().userName().get()
)
_num = _user.split("@")[0].split("-")[1] if _user.startswith("student-") else "01"
creds_scope = f"aws-course-creds-{_num}"

AWS_ACCESS_KEY_ID = dbutils.secrets.get(scope=creds_scope, key="aws-access-key-id")
AWS_SECRET_ACCESS_KEY = dbutils.secrets.get(scope=creds_scope, key="aws-secret-access-key")
AWS_REGION = dbutils.secrets.get(scope="aws-course-shared", key="aws-region")
os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_REGION"] = AWS_REGION
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION

print(f"Databricks user: {_user}  ->  scope {creds_scope}")
print("imports loaded: boto3, sagemaker, litellm, langfuse, pandas, pyarrow, strands")


In [ ]:
# Section 2 - identity, egress, Spark / Unity Catalog.
sts = boto3.client("sts", region_name=AWS_REGION)
spark_table = "bread_academy.course_data.fraud_transactions"

def _sts():
    ident = sts.get_caller_identity()
    assert ident["Account"] == "962804699607", "not the datacouch account"
    return ident["Arn"]

def _spark_read():
    n = spark.sql(f"SELECT COUNT(*) c FROM {spark_table}").collect()[0]["c"]
    assert n > 1000, f"fraud_transactions has only {n} rows"
    return f"{n:,} rows"

def _bedrock_egress():
    # any HTTP response proves the Databricks VNet reaches AWS public endpoints
    import urllib.request
    try:
        urllib.request.urlopen("https://sts.amazonaws.com", timeout=5)
    except urllib.error.HTTPError:
        pass  # an HTTP error is still a reachable endpoint
    return "VNet egress OK"

check("STS get_caller_identity (datacouch)", _sts)
check("Spark read bread_academy.course_data.fraud_transactions", _spark_read)
check("Network egress to AWS", _bedrock_egress)


In [ ]:
# Section 3 - S3 download + upload + delete round-trip on the course bucket.
s3 = boto3.client("s3", region_name=AWS_REGION)
S3_BUCKET = dbutils.secrets.get(scope="aws-course-shared", key="course-s3-bucket")
_probe_key = f"preflight/_probe_{_num}_{int(time.time())}.txt"

def _s3_head():
    s3.head_bucket(Bucket=S3_BUCKET)
    return S3_BUCKET

def _s3_download():
    # the Model Monitor baseline CSV bootstrap.py uploaded
    obj = s3.get_object(
        Bucket=S3_BUCKET, Key="fraud-classifier/training/baseline.csv"
    )
    size = len(obj["Body"].read())
    return f"baseline.csv {size} bytes"

def _s3_upload():
    s3.put_object(Bucket=S3_BUCKET, Key=_probe_key, Body=b"preflight probe\n")
    return _probe_key

def _s3_delete():
    s3.delete_object(Bucket=S3_BUCKET, Key=_probe_key)
    return "temp probe object removed"

check("S3 head_bucket bread-academy-shared", _s3_head)
check("S3 download (baseline.csv)", _s3_download)
check("S3 upload (temp probe key)", _s3_upload)
check("S3 delete (temp probe key)", _s3_delete)


In [ ]:
# Section 4 - SageMaker control-plane (Weeks 19-20).
sm = boto3.client("sagemaker", region_name=AWS_REGION)

def _sm_list_training():
    sm.list_training_jobs(MaxResults=1)
    return "list_training_jobs OK"

def _sm_describe_endpoint():
    r = sm.describe_endpoint(EndpointName="fraud-classifier-endpoint")
    return f"fraud-classifier-endpoint {r['EndpointStatus']}"

def _sm_list_mpg():
    sm.list_model_package_groups(MaxResults=1)
    return "list_model_package_groups OK"

def _sm_passrole():
    # the exec role the notebooks pass to SageMaker must be readable
    role = dbutils.secrets.get(
        scope="aws-course-shared", key="sagemaker-execution-role-arn"
    )
    return role.split("/")[-1]

check("SageMaker list_training_jobs", _sm_list_training)
check("SageMaker describe_endpoint fraud-classifier-endpoint", _sm_describe_endpoint)
check("SageMaker list_model_package_groups", _sm_list_mpg)
check("SageMaker execution-role-arn secret", _sm_passrole)


In [ ]:
# Section 5 - Bedrock LLM + Bedrock Knowledge Base.
# The KB check FAILS until the instructor provisioning script (build_kb.py)
# has created the datacouch KB and written knowledge-base-id into
# aws-course-shared. That is expected before the script is run.
bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)
bedrock_agent = boto3.client("bedrock-agent", region_name=AWS_REGION)
bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name=AWS_REGION)
BEDROCK_MODEL_ID = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

def _bedrock_converse():
    r = bedrock_runtime.converse(
        modelId=BEDROCK_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": "ping"}]}],
        inferenceConfig={"maxTokens": 5, "temperature": 0},
    )
    return r["output"]["message"]["content"][0]["text"].strip()

def _kb_id():
    kb = dbutils.secrets.get(scope="aws-course-shared", key="knowledge-base-id")
    assert kb and kb.upper() != "PENDING", "knowledge-base-id secret not set"
    return kb

def _kb_exists():
    kb = dbutils.secrets.get(scope="aws-course-shared", key="knowledge-base-id")
    r = bedrock_agent.get_knowledge_base(knowledgeBaseId=kb)
    return f"{r['knowledgeBase']['name']} {r['knowledgeBase']['status']}"

def _kb_retrieve():
    kb = dbutils.secrets.get(scope="aws-course-shared", key="knowledge-base-id")
    r = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=kb,
        retrievalQuery={"text": "wire transfer fraud policy"},
        retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": 2}},
    )
    return f"{len(r.get('retrievalResults', []))} chunks"

check("Bedrock converse (Sonnet 4.5)", _bedrock_converse)
check("Bedrock KB id secret present", _kb_id)
check("Bedrock KB get_knowledge_base", _kb_exists)
check("Bedrock KB retrieve", _kb_retrieve)


In [ ]:
# Section 6 - MLflow (Databricks-native), CloudWatch, SNS.
import mlflow
cloudwatch = boto3.client("cloudwatch", region_name=AWS_REGION)
sns = boto3.client("sns", region_name=AWS_REGION)

def _mlflow():
    mlflow.set_tracking_uri("databricks")
    exp = f"/Users/{_user}/pre-env-probe"
    mlflow.set_experiment(exp)
    with mlflow.start_run(run_name=f"probe-{int(time.time())}"):
        mlflow.log_metric("ok", 1.0)
    return exp

def _cw_describe():
    cloudwatch.describe_alarms(MaxRecords=1)
    return "describe_alarms OK"

def _cw_putalarm():
    # create then delete a throwaway alarm - proves PutMetricAlarm
    name = f"preflight-probe-{_num}"
    cloudwatch.put_metric_alarm(
        AlarmName=name, MetricName="Invocations", Namespace="AWS/SageMaker",
        Statistic="Sum", Period=60, EvaluationPeriods=1, Threshold=1e9,
        ComparisonOperator="GreaterThanThreshold", TreatMissingData="notBreaching",
    )
    cloudwatch.delete_alarms(AlarmNames=[name])
    return "put + delete alarm OK"

def _sns():
    arn = dbutils.secrets.get(scope="aws-course-shared", key="sns-alerts-topic-arn")
    sns.get_topic_attributes(TopicArn=arn)
    return arn.split(":")[-1]

check("MLflow set_experiment + log_metric (Databricks-native)", _mlflow)
check("CloudWatch describe_alarms", _cw_describe)
check("CloudWatch put + delete alarm", _cw_putalarm)
check("SNS get_topic_attributes (shared topic)", _sns)


In [ ]:
# Section 7 - MWAA Airflow (Weeks 21-22).
mwaa = boto3.client("mwaa", region_name=AWS_REGION)
MWAA_ENV = "bread-academy-airflow"

def _mwaa_get():
    r = mwaa.get_environment(Name=MWAA_ENV)
    return f"{MWAA_ENV} {r['Environment']['Status']}"

def _mwaa_invoke():
    # invoke_rest_api proves the airflow:InvokeRestApi grant works
    r = mwaa.invoke_rest_api(Name=MWAA_ENV, Method="GET", Path="/dags")
    code = r.get("RestApiStatusCode")
    assert code == 200, f"REST API returned {code}"
    return "invoke_rest_api /dags 200"

check("MWAA get_environment", _mwaa_get)
check("MWAA invoke_rest_api (airflow:InvokeRestApi)", _mwaa_invoke)


In [ ]:
# Section 8 - summary.
passed = sum(1 for ok, _ in _results if ok)
failed = [name for ok, name in _results if not ok]

print("=" * 60)
print(f"PRE-ENV PREFLIGHT: {passed}/{len(_results)} checks passed")
print("=" * 60)
if failed:
    print("\nFAILED checks:")
    for name in failed:
        print("  -", name)
    print("\nNote: the 3 Bedrock KB checks fail until the instructor runs")
    print("scripts/build_kb.py to create the datacouch KB. Any other")
    print("failure is a real environment gap to fix before class.")
else:
    print("\nAll preflight checks passed - environment ready.")
